# Flight Distance Calculation

This notebook calculates the great circle distance between departure and arrival airports for flight data using the Haversine formula.

In [11]:
# Import required libraries
import pandas as pd
import numpy as np
from pathlib import Path
import math
from typing import Tuple, Optional
from datetime import datetime, timedelta

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [12]:
# Load airport coordinates data
data_path = Path("../data")
airport_coords_df = pd.read_csv(data_path / "airport_coordinates.csv")

print(f"✅ Airport coordinates loaded: {len(airport_coords_df)} airports")
print(f"📊 Airports with coordinates: {len(airport_coords_df[airport_coords_df['latitude'].notna()])}")
print(f"\n🔍 First few airports:")
print(airport_coords_df.head())

✅ Airport coordinates loaded: 140 airports
📊 Airports with coordinates: 140

🔍 First few airports:
  iata_code   latitude  longitude     source
0       1HR  60.318769  24.968125  nominatim
1       1SN  60.318769  24.968125  nominatim
2       3SK  65.699722   7.650833    openaip
3       AAL  57.086435   9.867982  nominatim
4       AAR  56.304728  10.612299  nominatim


In [13]:
# Create airport coordinates lookup dictionary for fast access
airport_coords = {}
valid_coords = airport_coords_df[airport_coords_df['latitude'].notna()]

for _, row in valid_coords.iterrows():
    airport_coords[row['iata_code']] = {
        'latitude': row['latitude'],
        'longitude': row['longitude']
    }

print(f"✅ Airport coordinates lookup created for {len(airport_coords)} airports")
print(f"🔍 Sample lookup: OSL -> {airport_coords.get('OSL', 'Not found')}")

✅ Airport coordinates lookup created for 140 airports
🔍 Sample lookup: OSL -> {'latitude': 60.1978654, 'longitude': 11.0996754}


In [14]:
def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """
    Calculate the great circle distance between two points on Earth using the Haversine formula.
    
    Args:
        lat1, lon1: Latitude and longitude of first point in decimal degrees
        lat2, lon2: Latitude and longitude of second point in decimal degrees
        
    Returns:
        Distance in kilometers
    """
    # Convert latitude and longitude from degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    
    # Radius of Earth in kilometers
    R = 6371.0
    
    # Calculate the distance
    distance = R * c
    
    return distance

# Test the function
print("🧪 Testing Haversine distance function:")
# Oslo (OSL) to Bergen (BGO) - should be around 300km
osl_coords = airport_coords.get('OSL')
bgo_coords = airport_coords.get('BGO')

if osl_coords and bgo_coords:
    test_distance = haversine_distance(
        osl_coords['latitude'], osl_coords['longitude'],
        bgo_coords['latitude'], bgo_coords['longitude']
    )
    print(f"✅ OSL to BGO distance: {test_distance:.2f} km")
else:
    print("❌ Could not find coordinates for OSL or BGO")

print("✅ Haversine distance function created")

🧪 Testing Haversine distance function:
✅ OSL to BGO distance: 324.54 km
✅ Haversine distance function created


In [15]:
def calculate_flight_distance(dep_airport: str, arr_airport: str) -> Optional[float]:
    """
    Calculate the distance between two airports using their IATA codes.
    
    Args:
        dep_airport: Departure airport IATA code
        arr_airport: Arrival airport IATA code
        
    Returns:
        Distance in kilometers, or None if coordinates not available
    """
    # Get coordinates for both airports
    dep_coords = airport_coords.get(dep_airport)
    arr_coords = airport_coords.get(arr_airport)
    
    # Check if both airports have coordinates
    if not dep_coords or not arr_coords:
        return None
    
    # Calculate distance
    distance = haversine_distance(
        dep_coords['latitude'], dep_coords['longitude'],
        arr_coords['latitude'], arr_coords['longitude']
    )
    
    return distance

# Test the function
print("🧪 Testing flight distance calculation:")
test_routes = [
    ('OSL', 'BGO'),  # Oslo to Bergen
    ('OSL', 'TRD'),  # Oslo to Trondheim
    ('OSL', 'CPH'),  # Oslo to Copenhagen
    ('OSL', 'ARN'),  # Oslo to Stockholm
]

for dep, arr in test_routes:
    distance = calculate_flight_distance(dep, arr)
    if distance:
        print(f"✅ {dep} → {arr}: {distance:.2f} km")
    else:
        print(f"❌ {dep} → {arr}: Could not calculate distance")

print("\n✅ Flight distance calculation function created")

🧪 Testing flight distance calculation:
✅ OSL → BGO: 324.54 km
✅ OSL → TRD: 362.27 km
✅ OSL → CPH: 518.37 km
✅ OSL → ARN: 385.75 km

✅ Flight distance calculation function created


In [16]:
# Load historical flights data
print("📊 Loading historical flights data...")
df_flights = pd.read_csv(data_path / "avinor_data/historical_flights.csv")

print(f"✅ Historical flights loaded: {len(df_flights):,} flights")
print(f"📋 Columns: {list(df_flights.columns)}")
print(f"\n🔍 Sample data:")
print(df_flights[['flight_id', 'dep_airport', 'arr_airport']].head())

📊 Loading historical flights data...
✅ Historical flights loaded: 410,442 flights
📋 Columns: ['flight_id', 'dep_airport', 'dep_airport_group', 'arr_airport', 'arr_airport_group', 'service_type', 'std', 'sta', 'cancelled', 'atd', 'ata']

🔍 Sample data:
  flight_id dep_airport arr_airport
0     WF149         HOV         OSL
1     WF722         OSL         MJF
2     WF188         FDE         OSL
3     WF176         HOV         OSL
4     WF148         HOV         OSL


In [17]:
def calculate_flight_time(std: str, sta: str) -> Optional[float]:
    """
    Calculate expected flight time from scheduled departure and arrival times.
    
    Args:
        std: Scheduled Time of Departure (string format)
        sta: Scheduled Time of Arrival (string format)
        
    Returns:
        Flight time in minutes, or None if parsing fails
    """
    try:
        # Parse datetime strings - handle different formats
        if pd.isna(std) or pd.isna(sta):
            return None
            
        # Convert to pandas datetime for robust parsing
        std_dt = pd.to_datetime(std)
        sta_dt = pd.to_datetime(sta)
        
        # Calculate time difference
        time_diff = sta_dt - std_dt
        
        # Convert to minutes
        flight_time_minutes = time_diff.total_seconds() / 60
        
        # Handle edge cases (negative times might indicate next day arrival)
        if flight_time_minutes < 0:
            # Add 24 hours for next day arrival
            flight_time_minutes += 24 * 60
            
        # Sanity check: flight times should be reasonable (0.5 to 24 hours)
        if flight_time_minutes < 30 or flight_time_minutes > 24 * 60:
            return None
            
        return flight_time_minutes
        
    except Exception as e:
        return None

def calculate_flight_time_hours(std: str, sta: str) -> Optional[float]:
    """
    Calculate expected flight time in hours.
    
    Args:
        std: Scheduled Time of Departure (string format)
        sta: Scheduled Time of Arrival (string format)
        
    Returns:
        Flight time in hours, or None if parsing fails
    """
    minutes = calculate_flight_time(std, sta)
    if minutes is not None:
        return minutes / 60
    return None

# Test the function with sample data
print("🧪 Testing flight time calculation:")
test_times = [
    ("2018-01-02T16:40:00.000000", "2018-01-02T17:15:00.000000"),  # 35 minutes
    ("2018-01-28T13:04:00.000000", "2018-01-28T14:50:00.000000"),  # 1h 46m
    ("2018-04-07T23:30:00.000000", "2018-04-08T01:15:00.000000"),  # Overnight flight
]

for std, sta in test_times:
    flight_time = calculate_flight_time(std, sta)
    flight_time_hours = calculate_flight_time_hours(std, sta)
    if flight_time is not None:
        print(f"✅ {std} → {sta}")
        print(f"   Flight time: {flight_time:.0f} minutes ({flight_time_hours:.2f} hours)")
    else:
        print(f"❌ Could not calculate flight time for {std} → {sta}")

print("\n✅ Flight time calculation functions created")

🧪 Testing flight time calculation:
✅ 2018-01-02T16:40:00.000000 → 2018-01-02T17:15:00.000000
   Flight time: 35 minutes (0.58 hours)
✅ 2018-01-28T13:04:00.000000 → 2018-01-28T14:50:00.000000
   Flight time: 106 minutes (1.77 hours)
✅ 2018-04-07T23:30:00.000000 → 2018-04-08T01:15:00.000000
   Flight time: 105 minutes (1.75 hours)

✅ Flight time calculation functions created


# Flight Time Calculation

Calculate expected flight time based on scheduled departure and arrival times.

In [20]:
def add_distance_and_time_to_flights(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add distance and flight time calculations to flight data.
    
    Args:
        df: DataFrame with 'dep_airport', 'arr_airport', 'std', and 'sta' columns
        
    Returns:
        DataFrame with added 'distance_km', 'flight_time_minutes', and 'flight_time_hours' columns
    """
    print("🧮 Calculating distances and flight times for all flights...")
    
    # Create a copy to avoid modifying original
    df_enhanced = df.copy()
    
    # Calculate distances and flight times
    distances = []
    flight_times_minutes = []
    flight_times_hours = []
    missing_coords_count = 0
    missing_times_count = 0
    
    for idx, row in df_enhanced.iterrows():
        if idx % 50000 == 0:  # Progress indicator
            print(f"   📍 Processed {idx:,} flights...")
            
        # Calculate distance
        distance = calculate_flight_distance(row['dep_airport'], row['arr_airport'])
        distances.append(distance)
        if distance is None:
            missing_coords_count += 1
            
        # Calculate flight time
        flight_time_min = calculate_flight_time(row['std'], row['sta'])
        flight_time_hr = calculate_flight_time_hours(row['std'], row['sta'])
        flight_times_minutes.append(flight_time_min)
        flight_times_hours.append(flight_time_hr)
        if flight_time_min is None:
            missing_times_count += 1
    
    # Add new columns
    df_enhanced['distance_km'] = distances
    df_enhanced['flight_time_minutes'] = flight_times_minutes
    df_enhanced['flight_time_hours'] = flight_times_hours
    
    # Summary statistics
    valid_distances = df_enhanced['distance_km'].dropna()
    valid_times = df_enhanced['flight_time_minutes'].dropna()
    
    print(f"\\n📊 CALCULATION SUMMARY:")
    print(f"   • Total flights: {len(df_enhanced):,}")
    print(f"   • Distances calculated: {len(valid_distances):,}")
    print(f"   • Flight times calculated: {len(valid_times):,}")
    print(f"   • Missing coordinates: {missing_coords_count:,}")
    print(f"   • Missing time data: {missing_times_count:,}")
    print(f"   • Distance success rate: {(len(valid_distances)/len(df_enhanced)*100):.1f}%")
    print(f"   • Flight time success rate: {(len(valid_times)/len(df_enhanced)*100):.1f}%")
    
    if len(valid_distances) > 0:
        print(f"\\n🌍 DISTANCE STATISTICS:")
        print(f"   • Average distance: {valid_distances.mean():.2f} km")
        print(f"   • Median distance: {valid_distances.median():.2f} km")
        print(f"   • Min distance: {valid_distances.min():.2f} km")
        print(f"   • Max distance: {valid_distances.max():.2f} km")
        print(f"   • Standard deviation: {valid_distances.std():.2f} km")

    if len(valid_times) > 0:
        print(f"\\n⏱️ FLIGHT TIME STATISTICS:")
        print(f"   • Average flight time: {valid_times.mean():.1f} minutes ({valid_times.mean()/60:.2f} hours)")
        print(f"   • Median flight time: {valid_times.median():.1f} minutes ({valid_times.median()/60:.2f} hours)")
        print(f"   • Min flight time: {valid_times.min():.1f} minutes ({valid_times.min()/60:.2f} hours)")
        print(f"   • Max flight time: {valid_times.max():.1f} minutes ({valid_times.max()/60:.2f} hours)")
        print(f"   • Standard deviation: {valid_times.std():.1f} minutes")

    return df_enhanced

print("✅ Enhanced calculation function for datasets created")

✅ Enhanced calculation function for datasets created


In [21]:
# Calculate distances and flight times for the entire dataset
print("🚀 Starting enhanced calculation for all historical flights...")
print("This will calculate both distances and expected flight times...")
print("This may take a few moments...\n")

df_flights_enhanced = add_distance_and_time_to_flights(df_flights)

print("\n✅ Enhanced calculation completed!")

🚀 Starting enhanced calculation for all historical flights...
This will calculate both distances and expected flight times...
This may take a few moments...

🧮 Calculating distances and flight times for all flights...
   📍 Processed 0 flights...
   📍 Processed 50,000 flights...
   📍 Processed 100,000 flights...
   📍 Processed 150,000 flights...
   📍 Processed 200,000 flights...
   📍 Processed 250,000 flights...
   📍 Processed 300,000 flights...
   📍 Processed 350,000 flights...
   📍 Processed 400,000 flights...
\n📊 CALCULATION SUMMARY:
   • Total flights: 410,442
   • Distances calculated: 410,442
   • Flight times calculated: 278,839
   • Missing coordinates: 0
   • Missing time data: 131,603
   • Distance success rate: 100.0%
   • Flight time success rate: 67.9%
\n🌍 DISTANCE STATISTICS:
   • Average distance: 437.69 km
   • Median distance: 167.19 km
   • Min distance: 0.00 km
   • Max distance: 8160.64 km
   • Standard deviation: 1111.35 km
\n⏱️ FLIGHT TIME STATISTICS:
   • Average 

In [22]:
# Analyze the enhanced results
print("📈 DETAILED ANALYSIS OF FLIGHT DISTANCES AND TIMES:")
print("=" * 60)

# Show sample results
print("\n🔍 Sample flights with calculated distances and times:")
sample_enhanced = df_flights_enhanced[['flight_id', 'dep_airport', 'arr_airport', 'distance_km', 'flight_time_hours']].dropna().head(10)
print(sample_enhanced.to_string(index=False))

# Show flights without distance or time data
missing_data = df_flights_enhanced[
    (df_flights_enhanced['distance_km'].isna()) | 
    (df_flights_enhanced['flight_time_minutes'].isna())
]
if len(missing_data) > 0:
    print(f"\n❌ Flights with missing data ({len(missing_data)} total):")
    unique_missing = missing_data[['dep_airport', 'arr_airport', 'std', 'sta']].head(5)
    print(unique_missing.to_string(index=False))

# Distance and time distribution analysis
valid_distances = df_flights_enhanced['distance_km'].dropna()
valid_times = df_flights_enhanced['flight_time_minutes'].dropna()

if len(valid_distances) > 0:
    print(f"\n📊 DISTANCE DISTRIBUTION:")
    
    # Define distance categories
    categories = [
        (0, 500, "Short-haul (< 500 km)"),
        (500, 1500, "Medium-haul (500-1500 km)"),
        (1500, 4000, "Long-haul (1500-4000 km)"),
        (4000, float('inf'), "Ultra long-haul (> 4000 km)")
    ]
    
    for min_dist, max_dist, category_name in categories:
        count = len(valid_distances[(valid_distances >= min_dist) & (valid_distances < max_dist)])
        percentage = (count / len(valid_distances)) * 100
        print(f"   • {category_name}: {count:,} flights ({percentage:.1f}%)")

if len(valid_times) > 0:
    print(f"\n⏱️ FLIGHT TIME DISTRIBUTION:")
    
    # Define time categories
    time_categories = [
        (0, 60, "Very short (< 1 hour)"),
        (60, 180, "Short (1-3 hours)"),
        (180, 360, "Medium (3-6 hours)"),
        (360, 720, "Long (6-12 hours)"),
        (720, float('inf'), "Ultra long (> 12 hours)")
    ]
    
    for min_time, max_time, category_name in time_categories:
        count = len(valid_times[(valid_times >= min_time) & (valid_times < max_time)])
        percentage = (count / len(valid_times)) * 100
        print(f"   • {category_name}: {count:,} flights ({percentage:.1f}%)")
    
    # Top 10 longest flights by time
    print(f"\n🕐 TOP 10 LONGEST FLIGHTS BY TIME:")
    longest_time = df_flights_enhanced.nlargest(10, 'flight_time_minutes')[['dep_airport', 'arr_airport', 'flight_time_hours', 'distance_km']]
    for _, row in longest_time.iterrows():
        print(f"   • {row['dep_airport']} → {row['arr_airport']}: {row['flight_time_hours']:.2f} hours ({row['distance_km']:.0f} km)")
    
    # Top 10 shortest flights by time (excluding very short ones)
    print(f"\n⚡ TOP 10 SHORTEST FLIGHTS BY TIME:")
    shortest_time = df_flights_enhanced[df_flights_enhanced['flight_time_minutes'] > 30].nsmallest(10, 'flight_time_minutes')[['dep_airport', 'arr_airport', 'flight_time_hours', 'distance_km']]
    for _, row in shortest_time.iterrows():
        print(f"   • {row['dep_airport']} → {row['arr_airport']}: {row['flight_time_hours']:.2f} hours ({row['distance_km']:.0f} km)")

# Analyze speed (distance/time relationship)
flights_with_both = df_flights_enhanced.dropna(subset=['distance_km', 'flight_time_hours'])
if len(flights_with_both) > 0:
    flights_with_both['average_speed_kmh'] = flights_with_both['distance_km'] / flights_with_both['flight_time_hours']
    
    print(f"\n✈️ AVERAGE FLIGHT SPEED ANALYSIS:")
    print(f"   • Flights with both distance and time data: {len(flights_with_both):,}")
    speed_stats = flights_with_both['average_speed_kmh'].describe()
    print(f"   • Average speed: {speed_stats['mean']:.1f} km/h")
    print(f"   • Median speed: {speed_stats['50%']:.1f} km/h")
    print(f"   • Speed range: {speed_stats['min']:.1f} - {speed_stats['max']:.1f} km/h")
    
    # Add speed column to main dataset
    df_flights_enhanced['average_speed_kmh'] = df_flights_enhanced['distance_km'] / df_flights_enhanced['flight_time_hours']

📈 DETAILED ANALYSIS OF FLIGHT DISTANCES AND TIMES:

🔍 Sample flights with calculated distances and times:
flight_id dep_airport arr_airport  distance_km  flight_time_hours
    WF149         HOV         OSL   347.920342           0.583333
    WF722         OSL         MJF   630.128967           1.766667
    WF188         FDE         OSL   318.521863           1.000000
    WF176         HOV         OSL   347.920342           1.083333
    WF148         HOV         OSL   347.920342           1.016667
    WF145         SDN         OSL  5484.997723           0.916667
    WF781         OSY         OSL   475.915001           1.333333
    WF166         HOV         OSL   347.920342           1.000000
   SK4517         MOL         TRF   427.543536           2.916667
    WF176         SDN         OSL  5484.997723           1.000000

❌ Flights with missing data (131603 total):
dep_airport arr_airport                        std                        sta
        OSL         VDS 2018-12-19T08:15:00.0

/var/folders/mh/yf36fnwd79vbr3b1xg7_ctvc0000gn/T/ipykernel_43510/3055570450.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  flights_with_both['average_speed_kmh'] = flights_with_both['distance_km'] / flights_with_both['flight_time_hours']


In [23]:
# Save the enhanced dataset with distances and flight times
print("💾 Saving flight data with calculated distances and flight times...")

# Save to new CSV file
output_file = data_path / "historical_flights_enhanced.csv"
df_flights_enhanced.to_csv(output_file, index=False)

print(f"✅ Enhanced dataset saved to: {output_file}")
print(f"📊 Dataset contains {len(df_flights_enhanced):,} flights with enhanced calculations")

# Show final summary
distance_count = len(df_flights_enhanced[df_flights_enhanced['distance_km'].notna()])
time_count = len(df_flights_enhanced[df_flights_enhanced['flight_time_minutes'].notna()])
both_count = len(df_flights_enhanced.dropna(subset=['distance_km', 'flight_time_minutes']))

print(f"\n🎯 FINAL SUMMARY:")
print(f"   • Original flights: {len(df_flights):,}")
print(f"   • Flights with distance: {distance_count:,} ({(distance_count/len(df_flights)*100):.1f}%)")
print(f"   • Flights with time data: {time_count:,} ({(time_count/len(df_flights)*100):.1f}%)")
print(f"   • Flights with both distance and time: {both_count:,} ({(both_count/len(df_flights)*100):.1f}%)")
print(f"\n📋 NEW COLUMNS ADDED:")
print(f"   • 'distance_km': Great circle distance between airports")
print(f"   • 'flight_time_minutes': Expected flight time in minutes")
print(f"   • 'flight_time_hours': Expected flight time in hours")
print(f"   • 'average_speed_kmh': Average flight speed (distance/time)")
print(f"\n✅ Enhanced data ready for analysis and modeling")

💾 Saving flight data with calculated distances and flight times...
✅ Enhanced dataset saved to: ../data/historical_flights_enhanced.csv
📊 Dataset contains 410,442 flights with enhanced calculations

🎯 FINAL SUMMARY:
   • Original flights: 410,442
   • Flights with distance: 410,442 (100.0%)
   • Flights with time data: 278,839 (67.9%)
   • Flights with both distance and time: 278,839 (67.9%)

📋 NEW COLUMNS ADDED:
   • 'distance_km': Great circle distance between airports
   • 'flight_time_minutes': Expected flight time in minutes
   • 'flight_time_hours': Expected flight time in hours
   • 'average_speed_kmh': Average flight speed (distance/time)

✅ Enhanced data ready for analysis and modeling


## Usage Examples

The enhanced distance and flight time calculation functionality can be used in several ways:

### 1. Calculate distance for individual flight routes:
```python
distance = calculate_flight_distance('OSL', 'BGO')
print(f"Distance: {distance:.2f} km")
```

### 2. Calculate expected flight time:
```python
flight_time = calculate_flight_time("2018-01-02T16:40:00.000000", "2018-01-02T17:15:00.000000")
print(f"Flight time: {flight_time:.0f} minutes")

# Or in hours
flight_time_hours = calculate_flight_time_hours("2018-01-02T16:40:00.000000", "2018-01-02T17:15:00.000000")
print(f"Flight time: {flight_time_hours:.2f} hours")
```

### 3. Add both distances and flight times to any flight dataset:
```python
# For any DataFrame with 'dep_airport', 'arr_airport', 'std', and 'sta' columns
df_enhanced = add_distance_and_time_to_flights(your_flight_data)
```

### 4. Use the Haversine formula directly:
```python
# For any two coordinate pairs
distance = haversine_distance(lat1, lon1, lat2, lon2)
```

### 5. Access the enhanced dataset:
```python
# The enhanced dataset includes:
# - distance_km: Great circle distance between airports
# - flight_time_minutes: Expected flight duration in minutes
# - flight_time_hours: Expected flight duration in hours  
# - average_speed_kmh: Average flight speed (calculated from distance/time)
```

## Key Features:
- **Accurate Distance Calculation**: Uses the Haversine formula for great circle distances
- **Robust Time Parsing**: Handles different datetime formats and edge cases (overnight flights)
- **Speed Analysis**: Automatically calculates average flight speeds
- **Comprehensive Statistics**: Provides detailed analysis of flight patterns
- **Error Handling**: Gracefully handles missing data and invalid times

The calculations use scheduled departure (STD) and scheduled arrival (STA) times to determine expected flight duration, which is useful for flight planning and analysis.